# 00 イントロ：Qwen3-4B とチャットしてみる

このノートブックでは **Qwen3-4B** を動かしながら、言語モデルの基本を体験します。

## このノートブックでやること

1. **環境確認** — デバイス (CPU / MPS / CUDA) とパッケージを確認する
2. **モデル読み込み** — Hugging Face から Tokenizer と Model を準備する
3. **チャット** — シングルターン・マルチターンで会話する
4. **Thinking モード** — 推論過程を `<think>` ブロックで観察する
5. **chat template** — special token がどこに入るかを確認する（参考）

## Qwen3-4B とは

- Alibaba が開発した **4B（40億）パラメータ**の言語モデル
- 中国語・英語・日本語など多言語対応
- **Thinking モード**（推論を段階的に考える）と **non-thinking モード** を切り替えられる
- 基本は non-thinking モード（`enable_thinking=False`）で使い、Section 4 で Thinking モードも試す

## 全体の流れ（Transformer の処理）

```
テキスト
  ↓ tokenizer.apply_chat_template()  # チャット形式のフォーマットに変換
トークン列
  ↓ model.generate()                 # Transformer が次のトークンを繰り返し予測
生成トークン列
  ↓ tokenizer.decode()               # トークンをテキストに戻す
応答テキスト
```

## 0. 環境セットアップ（3環境 自動切替: Colabだけ pip / path も自動）


In [ ]:
# 3環境(Mac/Win/Colab)を同一ファイルで動かすための判定。Colabのみ pip（Mac/Winはenvに在るのでskip）。
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "transformers==5.9.0" "accelerate==1.13.0"
    print("Colab: pip done")
else:
    print("local(Mac/Win): pip skip（env利用）")

---
## 1. 環境確認

In [ ]:
import sys
import logging
import torch
import transformers

# HuggingFace Hub の認証警告を抑制する（cache から読む場合は不要なため）
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print("device:", device)
print("dtype:", dtype)

---
## 2. モデル読み込み

Tokenizer と Model を Hugging Face cache から読み込みます。  
初回は自動でダウンロードされます（約 8 GB）。  
2回目以降は cache から読むので高速です。

### トークナイザー

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedModel

MODEL_ID = "Qwen/Qwen3-4B"

print("tokenizer 読み込み中...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("  語彙サイズ:", tokenizer.vocab_size)
print("  tokenizer クラス:", type(tokenizer).__name__)

**トークナイザーの動作を少しだけ見てみる**

In [ ]:
ids = tokenizer.encode("京都大学の情報学科")
print("token ids:", ids)
for i, tid in enumerate(ids):
    piece = tokenizer.decode([tid]) 
    print(f"{i:2d}: {tid:6d} -> '{piece}'")

### モデル

In [ ]:
print("model 読み込み中...")
model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    attn_implementation="eager",  # attention weights を取得できる実装
)
model.to(device)  # pyright: ignore[reportArgumentType]
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"  パラメータ数: {total_params / 1e9:.2f}B")
print(f"  モデルクラス: {type(model).__name__}")
print(f"  レイヤー数: {model.config.num_hidden_layers}")
print(f"  隠れ層の次元数: {model.config.hidden_size}")
print(f"  attention head 数: {model.config.num_attention_heads}")

**ちょっとだけモデルを見てみる**

In [ ]:
model

**Qwen3-4B アーキテクチャ 補足**

| 行 | 説明 |
|---|---|
| `Embedding(151936, 2560)` | 語彙 151,936 トークンを 2,560 次元ベクトルに変換 |
| `36 x Qwen3DecoderLayer` | 36 層の Transformer ブロックを積み重ねる |
| `Qwen3Attention` | Self-Attention：各トークンが全トークンを参照して文脈を把握 |
| `q_proj: Linear(2560→4096)` | Query 射影：32 heads × 128 次元 |
| `k_proj: Linear(2560→1024)` | Key 射影：8 heads × 128 次元（GQA — Query より head 数が少ない） |
| `v_proj: Linear(2560→1024)` | Value 射影：8 heads × 128 次元（GQA） |
| `o_proj: Linear(4096→2560)` | Attention 出力を元の 2,560 次元に戻す |
| `q_norm / k_norm` | Query・Key に対する RMS 正規化（QK Norm） |
| `Qwen3MLP` | Feed-Forward Network（FFN）：SwiGLU 活性化を使用 |
| `gate_proj / up_proj: Linear(2560→9728)` | SwiGLU の 2 つの入力線形層（中間次元 9,728） |
| `down_proj: Linear(9728→2560)` | FFN 出力を 2,560 次元に戻す |
| `input_layernorm` | Attention 前の RMS 正規化（Pre-Norm） |
| `post_attention_layernorm` | FFN 前の RMS 正規化（Pre-Norm） |
| `norm: Qwen3RMSNorm((2560,))` | 全層通過後の最終 RMS 正規化 |
| `rotary_emb: Qwen3RotaryEmbedding` | 位置エンコーディング（RoPE） |
| `lm_head: Linear(2560→151936)` | 隠れ状態 → 語彙分布（次トークンの確率を出力） |


### アーキテクチャを深堀りする

`inspect.getsource()` を使うと、pip install 版の Transformers でもソースコードを確認できます。

In [ ]:
import inspect
from transformers.models.qwen3.modeling_qwen3 import (
    Qwen3DecoderLayer, Qwen3Attention, Qwen3MLP
)

**各層の処理順（Qwen3DecoderLayer.forward）**

In [ ]:
print(inspect.getsource(Qwen3DecoderLayer.forward))

**アテンション（Qwen3Attention）**

In [ ]:
print(inspect.getsource(Qwen3Attention))

**FFN（Qwen3MLP）**

In [ ]:
print(inspect.getsource(Qwen3MLP))

---

## 3. チャット

### チャット関数

`chat()` 関数を定義します。

処理の流れ：
1. `messages`（ユーザーとアシスタントのやりとりリスト）を **chat template** でフォーマット
2. テキストを **トークン列**（整数の配列）に変換
3. `model.generate()` で次のトークンを繰り返し予測 → 応答トークン列
4. 生成部分だけを取り出してテキストにデコード

**`messages` の構造（`list[dict]`）**

`messages` は Python の「リスト」で、各要素が「辞書（dict）」になっています。

```python
# list：[ ] で囲まれた並び
# dict：{ } で囲まれたキーと値のペア

messages = [                                     # list の開始
    {"role": "user",      "content": "こんにちは"},  # dict（1つ目）
    {"role": "assistant", "content": "こんにちは！"}, # dict（2つ目）
    {"role": "user",      "content": "今日の天気は？"},# dict（3つ目）
]                                                # list の終了
```

各 `dict` には必ず2つのキーがあります：

| キー | 値 | 意味 |
|---|---|---|
| `"role"` | `"user"` または `"assistant"` | 誰の発言か |
| `"content"` | 任意のテキスト | 発言の内容 |

会話のターンが増えるたびに、この `list` に `dict` が追加されていきます。

In [ ]:
def chat(messages: list[dict],
            max_new_tokens: int = 256, enable_thinking: bool = False, skip_special_tokens: bool = True) -> str:
    """messages を受け取り、モデルの応答テキストを返す。"""
    # chat template を適用してプロンプト文字列を作る
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    # トークン化
    inputs = tokenizer(text, return_tensors="pt").to(device)

    # 生成
    with torch.no_grad():
        output_ids = model.generate(  # type: ignore[union-attr]
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # 入力部分を除いた生成トークンだけをデコード
    n_input = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0][n_input:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=skip_special_tokens)
    return response

### シングルターン

1 往復のシンプルな会話から始めます。

In [ ]:
# シングルターンの例
user_text = "言語モデルとは何か、1〜2文で教えてください。"
messages = [
    {"role": "user", "content": user_text}
]

response = chat(messages)
print("[User]", user_text)
print("[Assistant]", response)

### マルチターン

会話履歴（`messages` リスト）に応答を追加し続けることで、**文脈を保ったやりとり**ができます。

Transformer は入力トークン列全体を毎回処理するため、
過去の発言をリストに入れて渡すだけで文脈が維持されます。

In [ ]:
def chat_turn(messages: list[dict], user_text: str, **kwargs) -> list[dict]:
    """user_text を追加して応答を得る。更新済み messages を返す。"""
    messages = messages + [{"role": "user", "content": user_text}]
    response = chat(messages, **kwargs)
    messages = messages + [{"role": "assistant", "content": response}]
    return messages


def print_turn(history: list[dict]):
    print(f"[User]      {history[-2]['content']}")
    print(f"[Assistant] {history[-1]['content']}")
    print()

In [ ]:
# 1ターン目: 最初の質問
history = []
history = chat_turn(history, "トークンとは何ですか？")
print_turn(history)

In [ ]:
# 2ターン目：前の回答を受けて続ける
history = chat_turn(history, "具体的に「京都」という単語はいくつのトークンに分割されますか？")
print_turn(history)

In [ ]:
# 3ターン目
history = chat_turn(history, "ありがとう。では Qwen3 の語彙サイズはいくつですか？")
print_turn(history)

**ハルシネーションの確認**

モデルは「175,000語」と答えましたが、正しいでしょうか？`tokenizer.vocab_size` で実際の値を確認してみます。

In [ ]:
actual = tokenizer.vocab_size
print(f"実際の語彙サイズ: {actual:,}")

---

## 4. Thinking モード

Qwen3 には通常の応答モードに加えて、**Thinking モード**があります。
モデルが回答する前に `<think>...</think>` ブロックの中で推論過程を展開します。

`enable_thinking=True` に変えるだけで有効になります。
thinking 部分が長くなるため、`max_new_tokens` は大きめに設定します。

In [ ]:
user_text = "言語モデルとは何か、1〜2文で教えてください。"
messages = [{"role": "user", "content": user_text}]

response = chat(
    messages,
    max_new_tokens=512,
    enable_thinking=True,
    skip_special_tokens=False,  # <think> タグを見るため
)
print(response)

`skip_special_tokens=False` にしているため、通常は非表示になる special token も出力に含まれます。

| 出力 | 意味 |
|---|---|
| `<think>...</think>` | モデルの推論過程（Thinking モード） |
| `<\|im_end\|>` | アシスタントの発言終了を示す special token |

通常の chat（`skip_special_tokens=True`）では `<\|im_end\|>` は自動的に除去されます。

---

## 5. 参考：chat template の中身を見る

`apply_chat_template()` が何を作っているか確認します。
Special token（`<|im_start|>` など）がどこに入るかを見てみましょう。

In [ ]:
sample_messages = [
    {"role": "user", "content": "こんにちは"},
    {"role": "assistant", "content": "こんにちは！何かお手伝いできますか？"},
    {"role": "user", "content": "今日の天気は？"},
]

formatted = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(formatted)

In [ ]:
# トークン列も見てみる
token_ids = tokenizer.encode(formatted)
print(f"トークン数: {len(token_ids)}")
print()

for i, tid in enumerate(token_ids):
    piece = tokenizer.decode([tid])
    print(f"  [{i:2d}] id={tid:6d}  '{piece}'")

---
## まとめ

| ステップ | 処理 | 関数 |
|---|---|---|
| フォーマット | テキスト → chat template 形式 | `tokenizer.apply_chat_template()` |
| トークン化 | テキスト → token ID 列 | `tokenizer()` |
| 生成 | token ID 列 → 応答 token ID 列 | `model.generate()` |
| デコード | 応答 token ID 列 → テキスト | `tokenizer.decode()` |

次のノートブックでは **tokenizer** の動作を詳しく観察します。